In [1]:
from collections import deque
from datetime import datetime
import hashlib
import os
import re
import sqlite3
import time
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import pandas as pd
import requests

# ==========================================
# 1. CẤU HÌNH CRAWLER (PMLR)
# ==========================================
TOPIC = "Proceedings of Machine Learning Research (PMLR)"

SEED_URLS = [
    "https://proceedings.mlr.press/",  # Trang chủ chứa danh sách tất cả các Volume/Hội nghị
]

ALLOWED_DOMAINS = ["proceedings.mlr.press"]
MAX_DEPTH = 3
MAX_PAGES = 350  # Đặt mục tiêu 350 trang (dễ dàng vượt mốc 330 của bạn bạn)

# --- GIỚI HẠN THỜI GIAN VÀ TỐC ĐỘ ---
MAX_TIME_SECONDS = 480  # TỰ ĐỘNG NGẮT SAU 2 PHÚT (120 GIÂY)
CRAWL_DELAY = 0.05  # Tốc độ request siêu tốc 0.05s / request
REQUEST_TIMEOUT = 5
DB_PATH = "data/pmlr_crawler.db"

# Các đuôi file không phải trang HTML
IGNORED_EXTENSIONS = (
    ".pdf",
    ".ps",
    ".gz",
    ".tar",
    ".zip",
    ".jpg",
    ".png",
    ".gif",
    ".css",
    ".js",
    ".bib",
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

print("=== CRAWLER CONFIGURATION ===")
print(f"Topic          : {TOPIC}")
print(f"Target Pages   : {MAX_PAGES}")
print(f"Max Time Limit : {MAX_TIME_SECONDS} giây")
print(f"Crawl Delay    : {CRAWL_DELAY}s")


# ==========================================
# 2. URL FRONTIER & DUPLICATE DETECTOR
# ==========================================
class URLFrontier:

    def __init__(self, max_depth=3):
        self.queue = deque()
        self.visited = set()
        self.max_depth = max_depth
        self.discovered_count = 0
        self.skipped_count = 0

    def add_url(self, url, depth):
        self.discovered_count += 1
        if depth > self.max_depth or url in self.visited:
            self.skipped_count += 1
            return False
        self.visited.add(url)
        self.queue.append((url, depth))
        return True

    def get_next(self):
        return self.queue.popleft() if self.queue else None

    def is_empty(self):
        return len(self.queue) == 0


class DuplicateDetector:

    def __init__(self):
        self.fingerprints = set()

    def is_duplicate(self, text_content):
        if not text_content:
            return True
        fp = hashlib.sha256(
            text_content.strip().encode("utf-8")
        ).hexdigest()
        if fp in self.fingerprints:
            return True
        self.fingerprints.add(fp)
        return False


# ==========================================
# 3. QUẢN LÝ CƠ SỞ DỮ LIỆU SQLITE
# ==========================================
class Database:

    def __init__(self, db_path):
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        self.conn = sqlite3.connect(db_path)
        self.create_tables()

    def create_tables(self):
        cursor = self.conn.cursor()
        # Bảng 1: Pages (Dữ liệu trang web & bài báo khoa học)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT UNIQUE,
                domain TEXT,
                title TEXT,
                authors TEXT,
                raw_abstract TEXT,
                clean_abstract TEXT,
                is_scientific_paper INTEGER DEFAULT 0,
                depth INTEGER,
                status_code INTEGER,
                crawled_at TEXT
            )
        """)
        # Bảng 2: Links (Đồ thị liên kết web)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS links (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                source_url TEXT,
                target_url TEXT
            )
        """)
        self.conn.commit()

    def insert_page(
        self,
        url,
        domain,
        title,
        authors,
        raw_abstract,
        clean_abstract,
        is_scientific_paper,
        depth,
        status_code,
        crawled_at,
    ):
        cursor = self.conn.cursor()
        cursor.execute(
            """
            INSERT OR IGNORE INTO pages (url, domain, title, authors, raw_abstract, clean_abstract, is_scientific_paper, depth, status_code, crawled_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
            (
                url,
                domain,
                title,
                authors,
                raw_abstract,
                clean_abstract,
                is_scientific_paper,
                depth,
                status_code,
                crawled_at,
            ),
        )
        self.conn.commit()

    def insert_link(self, source_url, target_url):
        cursor = self.conn.cursor()
        cursor.execute(
            """
            INSERT INTO links (source_url, target_url)
            VALUES (?, ?)
        """,
            (source_url, target_url),
        )
        self.conn.commit()

    def close(self):
        self.conn.close()


# ==========================================
# 4. PARSER VÀ LÀM SẠCH DỮ LIỆU PMLR
# ==========================================
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"\$.*?\$", "", text)  # Bỏ công thức LaTeX
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_page_data(html_content, url):
    soup = BeautifulSoup(html_content, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else "No Title"
    authors = "Unknown"
    raw_abstract = ""
    is_scientific_paper = 0
    extracted_links = []

    # Nhận diện trang bài báo khoa học PMLR (kết thúc bằng .html và không phải index.html)
    if url.endswith(".html") and not url.endswith("index.html"):
        is_scientific_paper = 1

        # 1. Tiêu đề
        h1_tag = soup.find("h1")
        if h1_tag:
            title = h1_tag.get_text(strip=True)

        # 2. Tác giả
        authors_span = soup.find("span", class_="authors")
        if authors_span:
            authors = authors_span.get_text(strip=True)

        # 3. Tóm tắt (Abstract)
        abstract_div = soup.find("div", id="abstract")
        if abstract_div:
            raw_abstract = abstract_div.get_text(strip=True)

    clean_abstract = clean_text(
        raw_abstract if raw_abstract else title
    )

    # Trích xuất liên kết
    for tag in soup.find_all("a", href=True):
        href = tag["href"].strip()
        if href.startswith(("mailto:", "javascript:", "tel:", "#")):
            continue

        full_url = urljoin(url, href)
        parsed = urlparse(full_url)

        if (
            parsed.scheme in ("http", "https")
            and parsed.netloc.lower() in ALLOWED_DOMAINS
        ):
            path_lower = parsed.path.lower()
            if not any(ext in path_lower for ext in IGNORED_EXTENSIONS):
                clean_u = (
                    f"{parsed.scheme}://{parsed.netloc}{parsed.path}".rstrip(
                        "/"
                    )
                )
                extracted_links.append(clean_u)

    return (
        title,
        authors,
        raw_abstract,
        clean_abstract,
        is_scientific_paper,
        extracted_links,
    )


# ==========================================
# 5. THỰC THI CRAWLER
# ==========================================
def run_crawler():
    frontier = URLFrontier(max_depth=MAX_DEPTH)
    detector = DuplicateDetector()
    db = Database(DB_PATH)

    crawled_count = 0
    failed_requests = 0
    paper_count = 0
    non_paper_count = 0
    status_code_stats = {}
    depth_stats = {d: 0 for d in range(MAX_DEPTH + 1)}

    session = requests.Session()
    session.headers.update(HEADERS)

    for url in SEED_URLS:
        frontier.add_url(url, depth=0)

    start_global_time = time.time()
    print(
        f"\n=== BẮT ĐẦU CRAWL PMLR (SIÊU TỐC - TỰ ĐỘNG DỪNG KHI ĐỦ TRANG HOẶC HẾT {MAX_TIME_SECONDS}s) ==="
    )

    while not frontier.is_empty() and crawled_count < MAX_PAGES:
        # Kiểm tra điều kiện thời gian tối đa
        elapsed_global = time.time() - start_global_time
        if elapsed_global >= MAX_TIME_SECONDS:
            print(
                f"\n[TIMEOUT] Đã đạt giới hạn {MAX_TIME_SECONDS}s. Tự động dừng crawler và chốt dữ liệu!"
            )
            break

        current_url, current_depth = frontier.get_next()

        try:
            response = session.get(current_url, timeout=REQUEST_TIMEOUT)
            status_code = response.status_code
            final_url = response.url
        except requests.RequestException:
            failed_requests += 1
            continue

        status_code_stats[status_code] = (
            status_code_stats.get(status_code, 0) + 1
        )

        if status_code == 200:
            response.encoding = "utf-8"
            (
                title,
                authors,
                raw_abstract,
                clean_abstract,
                is_scientific_paper,
                extracted_links,
            ) = extract_page_data(response.text, final_url)

            content_to_check = (
                clean_abstract if clean_abstract else title
            )
            if detector.is_duplicate(content_to_check):
                continue

            crawled_count += 1
            depth_stats[current_depth] = (
                depth_stats.get(current_depth, 0) + 1
            )
            crawled_at = datetime.now().isoformat()
            final_domain = urlparse(final_url).netloc.lower()

            if is_scientific_paper == 1:
                paper_count += 1
            else:
                non_paper_count += 1

            # 1. Lưu vào bảng pages
            db.insert_page(
                final_url,
                final_domain,
                title,
                authors,
                raw_abstract,
                clean_abstract,
                is_scientific_paper,
                current_depth,
                status_code,
                crawled_at,
            )

            # 2. Lưu vào bảng links
            for link in extracted_links:
                db.insert_link(final_url, link)
                frontier.add_url(link, depth=current_depth + 1)

            if crawled_count % 20 == 0 or crawled_count == MAX_PAGES:
                print(
                    f"Progress: [{crawled_count:03d}/{MAX_PAGES}] | Time: {elapsed_global:.1f}s | Papers (1): {paper_count} | Volume/Index (0): {non_paper_count}"
                )
        else:
            failed_requests += 1

        time.sleep(CRAWL_DELAY)

    total_time = time.time() - start_global_time
    print("\n" + "=" * 25 + " CRAWLING SUMMARY " + "=" * 25)
    print(f"Tổng thời gian chạy    : {total_time:.2f} giây")
    print(f"Pages Crawled          : {crawled_count}")
    print(f"Scientific Papers (1)  : {paper_count}")
    print(f"Volume/Index Pages(0)  : {non_paper_count}")
    print(f"Unique URLs Discovered : {frontier.discovered_count}")
    print(f"Skipped URLs           : {frontier.skipped_count}")
    print("=" * 68)

    return db


# ==========================================
# 6. KIỂM TRA BẢNG CƠ SỞ DỮ LIỆU SQLITE
# ==========================================
def verify_database(db):
    cursor = db.conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM pages")
    total_pages = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM links")
    total_links = cursor.fetchone()[0]

    cursor.execute(
        "SELECT COUNT(*) FROM pages WHERE is_scientific_paper = 1"
    )
    total_papers = cursor.fetchone()[0]

    print("\n=== KIỂM TRA CƠ SỞ DỮ LIỆU SQLITE ===")
    print(f"-> Bảng 'pages': {total_pages} bản ghi (Bài báo khoa học 1: {total_papers})")
    print(f"-> Bảng 'links': {total_links} liên kết")

    if total_pages > 0:
        df_pages = pd.read_sql_query(
            "SELECT id, is_scientific_paper, title, url FROM pages LIMIT 5",
            db.conn,
        )
        print("\nXem trước 5 bản ghi đầu tiên trong 'pages':")
        print(df_pages.to_string(index=False))


# ==========================================
# CHẠY CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    db = run_crawler()
    verify_database(db)
    db.close()
    print("\nĐã đóng kết nối cơ sở dữ liệu an toàn.")

=== CRAWLER CONFIGURATION ===
Topic          : Proceedings of Machine Learning Research (PMLR)
Target Pages   : 350
Max Time Limit : 480 giây (2 phút)
Crawl Delay    : 0.05s

=== BẮT ĐẦU CRAWL PMLR (SIÊU TỐC - TỰ ĐỘNG DỪNG KHI ĐỦ TRANG HOẶC HẾT 480s) ===


C:\Users\kiyor\AppData\Local\Temp\ipykernel_892\2049502793.py:210: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_content, "html.parser")


Progress: [020/350] | Time: 8.1s | Papers (1): 2 | Volume/Index (0): 18
Progress: [040/350] | Time: 18.8s | Papers (1): 2 | Volume/Index (0): 38
Progress: [060/350] | Time: 48.0s | Papers (1): 2 | Volume/Index (0): 58
Progress: [080/350] | Time: 70.8s | Papers (1): 2 | Volume/Index (0): 78
Progress: [100/350] | Time: 84.9s | Papers (1): 2 | Volume/Index (0): 98
Progress: [120/350] | Time: 110.8s | Papers (1): 2 | Volume/Index (0): 118
Progress: [140/350] | Time: 128.5s | Papers (1): 2 | Volume/Index (0): 138
Progress: [160/350] | Time: 152.5s | Papers (1): 2 | Volume/Index (0): 158
Progress: [180/350] | Time: 173.8s | Papers (1): 2 | Volume/Index (0): 178
Progress: [200/350] | Time: 191.9s | Papers (1): 2 | Volume/Index (0): 198
Progress: [220/350] | Time: 215.9s | Papers (1): 2 | Volume/Index (0): 218
Progress: [240/350] | Time: 234.1s | Papers (1): 2 | Volume/Index (0): 238
Progress: [260/350] | Time: 255.5s | Papers (1): 2 | Volume/Index (0): 258
Progress: [280/350] | Time: 273.0s |